# N-Hits Context Forecasting


In [ ]:
MODEL_DIR_NAME = 'N-Hits'
MODEL_LABEL = 'N-HiTS'
MODEL_TAG = 'nhits'

from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message=r".*'H' is deprecated.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=r".*'T' is deprecated.*", category=FutureWarning)

# Manual window length settings used by the final baseline experiments.
#
# Frequency | 24h target | 7d target | 28d target | 2d input | 7d input
# 30min     | 48         | 336       | 1344       | 96       | 336
# 1H        | 24         | 168       | 672        | 48       | 168
# 2H        | 12         | 84        | 336        | 24       | 84
# 3H        | 8          | 56        | 224        | 16       | 56
FREQ = "30min"
PRIMARY_HORIZON = "24h"
SEQ_LEN = 48
INPUT_LEN = 96
TRAIN_STRIDE = SEQ_LEN

SEED = 0
VAL_FRAC = 0.20
BATCH_SIZE = 64
EPOCHS = 10
EVAL_WINDOWS = 512
NUM_SAMPLES = 200
MAX_TRAIN_SAMPLES_PER_TS = 64
NUM_LOADER_WORKERS = 2 if sys.platform != "win32" else 0
QUANTILES = [0.1, 0.5, 0.9]
ALPHA_PI = 0.20
BINS = 80

HOUSEHOLD_ID_COL = "CUSTOMER_KEY"
TIME_COL = "READING_DATETIME"
TARGET_COL = "kWh"

STATIC_BASE_COLS = [
    "NUM_OCCUPANTS", "NUM_ROOMS_HEATED", "NUM_REFRIGERATORS",
    "Unit", "SemiDetached", "SeparateHouse",
    "HAS_GAS_HEATING", "HAS_GAS_HOT_WATER", "HAS_GAS_COOKING",
    "HAS_POOLPUMP", "Ducted", "SplitSystem", "NoAirCon", "OtherAirCon",
    "CONTROLLED_LOAD_CNT",
]

STATIC_CAT_CANDIDATES = [
    "NEAREST_STATION_NO", "STATION_NO", "station_id", "StationID",
    "TrialRegion", "TRIAL_REGION", "TrialRegionID", "trial_region_id",
]

PAST_COV_COLS = [
    "HourSin", "HourCos", "HalfHourSin", "HalfHourCos",
    "WeekdaySin", "WeekdayCos", "MonthSin", "MonthCos",
    "Summer", "Fall", "Winter", "Spring",
    "Temperature", "CDD", "HDD", "wind_speed",
    "Temperature_lag_2", "Temperature_lag_6",
    "Temperature_lag_48", "Temperature_lag_144",
    "temperature_lag_2", "temperature_lag_6",
    "temperature_lag_48", "temperature_lag_144",
]

FUTURE_COV_COLS = [
    "HourSin", "HourCos", "HalfHourSin", "HalfHourCos",
    "WeekdaySin", "WeekdayCos", "MonthSin", "MonthCos",
    "Summer", "Fall", "Winter", "Spring",
]

def pandas_freq(freq):
    return freq.replace("H", "h")


# All cleaned notebooks use one fixed dataset location.
SORTED_DIR = Path("..")
DATA_PATH = SORTED_DIR / "data_with_weather.pickle"
MODEL_DIR = SORTED_DIR / MODEL_DIR_NAME
print(f"MODEL_DIR={MODEL_DIR}")
print(
    f"FREQ={FREQ} | horizon={PRIMARY_HORIZON} | SEQ_LEN={SEQ_LEN} | "
    f"INPUT_LEN={INPUT_LEN} | TRAIN_STRIDE={TRAIN_STRIDE}"
)


### Preprocessing Roadmap

The preprocessing stage is intentionally a short pipeline rather than a model-specific trick:

1. Load the shared household energy table.
2. Keep the columns needed for this experiment and fill any missing optional covariates.
3. Convert timestamps, customer IDs, numeric covariates, and categorical IDs into model-ready types.
4. Add cyclical calendar features so hour, weekday, and month wrap around naturally.
5. Resample each customer to the selected frequency.
6. Split by customer so validation households are unseen during training.
7. Build context/target windows or library time-series objects, then scale the model inputs.

The small checks in this section are kept only where they prevent silent data leakage, missing-column errors, or invalid tensor shapes.


In [ ]:
# Place the dataset in the sorted directory, one level above this model notebook.
df = pd.read_pickle(DATA_PATH).copy()
print("Data file:", DATA_PATH)

required_columns = [TIME_COL, TARGET_COL, HOUSEHOLD_ID_COL]
missing_required = [column for column in required_columns if column not in df.columns]
if missing_required:
    raise KeyError(f"Dataset is missing required columns: {missing_required}")

# Cyclical calendar values are known for both the historical and forecast windows.
def add_calendar_covariates(frame, time_values):
    dt = pd.DatetimeIndex(pd.to_datetime(time_values))
    hour = dt.hour.astype(np.float32)
    minute = dt.minute.astype(np.float32)
    half_hour = hour * 2 + (minute // 30)
    weekday = dt.weekday.astype(np.float32)
    month = dt.month.astype(np.float32)

    frame["HourSin"] = np.sin(2.0 * np.pi * hour / 24.0).astype("float32")
    frame["HourCos"] = np.cos(2.0 * np.pi * hour / 24.0).astype("float32")
    frame["HalfHourSin"] = np.sin(2.0 * np.pi * half_hour / 48.0).astype("float32")
    frame["HalfHourCos"] = np.cos(2.0 * np.pi * half_hour / 48.0).astype("float32")
    frame["WeekdaySin"] = np.sin(2.0 * np.pi * weekday / 7.0).astype("float32")
    frame["WeekdayCos"] = np.cos(2.0 * np.pi * weekday / 7.0).astype("float32")
    frame["MonthSin"] = np.sin(2.0 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["MonthCos"] = np.cos(2.0 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Summer"] = np.isin(month, [12, 1, 2]).astype("float32")
    frame["Fall"] = np.isin(month, [3, 4, 5]).astype("float32")
    frame["Winter"] = np.isin(month, [6, 7, 8]).astype("float32")
    frame["Spring"] = np.isin(month, [9, 10, 11]).astype("float32")
    return frame


def static_cat_code_col(column):
    return "cat_" + re.sub(r"[^A-Za-z0-9]+", "_", column).strip("_")


# Keep the feature layout stable when an optional numerical field is absent.
if "NUM_OCCUPANTS" not in df.columns: df["NUM_OCCUPANTS"] = 0.0
if "NUM_ROOMS_HEATED" not in df.columns: df["NUM_ROOMS_HEATED"] = 0.0
if "NUM_REFRIGERATORS" not in df.columns: df["NUM_REFRIGERATORS"] = 0.0
if "Unit" not in df.columns: df["Unit"] = 0.0
if "SemiDetached" not in df.columns: df["SemiDetached"] = 0.0
if "SeparateHouse" not in df.columns: df["SeparateHouse"] = 0.0
if "HAS_GAS_HEATING" not in df.columns: df["HAS_GAS_HEATING"] = 0.0
if "HAS_GAS_HOT_WATER" not in df.columns: df["HAS_GAS_HOT_WATER"] = 0.0
if "HAS_GAS_COOKING" not in df.columns: df["HAS_GAS_COOKING"] = 0.0
if "HAS_POOLPUMP" not in df.columns: df["HAS_POOLPUMP"] = 0.0
if "Ducted" not in df.columns: df["Ducted"] = 0.0
if "SplitSystem" not in df.columns: df["SplitSystem"] = 0.0
if "NoAirCon" not in df.columns: df["NoAirCon"] = 0.0
if "OtherAirCon" not in df.columns: df["OtherAirCon"] = 0.0
if "CONTROLLED_LOAD_CNT" not in df.columns: df["CONTROLLED_LOAD_CNT"] = 0.0
if "Temperature" not in df.columns: df["Temperature"] = 0.0
temperature_for_degree_days = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0)
if "CDD" not in df.columns: df["CDD"] = np.maximum(temperature_for_degree_days - 18.0, 0.0).astype("float32")
if "HDD" not in df.columns: df["HDD"] = np.maximum(18.0 - temperature_for_degree_days, 0.0).astype("float32")
if "wind_speed" not in df.columns: df["wind_speed"] = 0.0
if "Temperature_lag_2" not in df.columns: df["Temperature_lag_2"] = 0.0
if "Temperature_lag_6" not in df.columns: df["Temperature_lag_6"] = 0.0
if "Temperature_lag_48" not in df.columns: df["Temperature_lag_48"] = 0.0
if "Temperature_lag_144" not in df.columns: df["Temperature_lag_144"] = 0.0
if "temperature_lag_2" not in df.columns: df["temperature_lag_2"] = 0.0
if "temperature_lag_6" not in df.columns: df["temperature_lag_6"] = 0.0
if "temperature_lag_48" not in df.columns: df["temperature_lag_48"] = 0.0
if "temperature_lag_144" not in df.columns: df["temperature_lag_144"] = 0.0

# Convert identifiers and timestamps before sorting or resampling.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df[HOUSEHOLD_ID_COL] = pd.to_numeric(df[HOUSEHOLD_ID_COL], errors="coerce").astype("int64")
df = df.sort_values([HOUSEHOLD_ID_COL, TIME_COL])
df = add_calendar_covariates(df, df[TIME_COL])

# Convert model inputs explicitly so the preprocessing can be read in execution order.
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0.0).astype("float32")
df["NUM_OCCUPANTS"] = pd.to_numeric(df["NUM_OCCUPANTS"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_ROOMS_HEATED"] = pd.to_numeric(df["NUM_ROOMS_HEATED"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_REFRIGERATORS"] = pd.to_numeric(df["NUM_REFRIGERATORS"], errors="coerce").fillna(0.0).astype("float32")
df["Unit"] = pd.to_numeric(df["Unit"], errors="coerce").fillna(0.0).astype("float32")
df["SemiDetached"] = pd.to_numeric(df["SemiDetached"], errors="coerce").fillna(0.0).astype("float32")
df["SeparateHouse"] = pd.to_numeric(df["SeparateHouse"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HEATING"] = pd.to_numeric(df["HAS_GAS_HEATING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HOT_WATER"] = pd.to_numeric(df["HAS_GAS_HOT_WATER"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_COOKING"] = pd.to_numeric(df["HAS_GAS_COOKING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_POOLPUMP"] = pd.to_numeric(df["HAS_POOLPUMP"], errors="coerce").fillna(0.0).astype("float32")
df["Ducted"] = pd.to_numeric(df["Ducted"], errors="coerce").fillna(0.0).astype("float32")
df["SplitSystem"] = pd.to_numeric(df["SplitSystem"], errors="coerce").fillna(0.0).astype("float32")
df["NoAirCon"] = pd.to_numeric(df["NoAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["OtherAirCon"] = pd.to_numeric(df["OtherAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["CONTROLLED_LOAD_CNT"] = pd.to_numeric(df["CONTROLLED_LOAD_CNT"], errors="coerce").fillna(0.0).astype("float32")
df["HourSin"] = pd.to_numeric(df["HourSin"], errors="coerce").fillna(0.0).astype("float32")
df["HourCos"] = pd.to_numeric(df["HourCos"], errors="coerce").fillna(0.0).astype("float32")
df["HalfHourSin"] = pd.to_numeric(df["HalfHourSin"], errors="coerce").fillna(0.0).astype("float32")
df["HalfHourCos"] = pd.to_numeric(df["HalfHourCos"], errors="coerce").fillna(0.0).astype("float32")
df["WeekdaySin"] = pd.to_numeric(df["WeekdaySin"], errors="coerce").fillna(0.0).astype("float32")
df["WeekdayCos"] = pd.to_numeric(df["WeekdayCos"], errors="coerce").fillna(0.0).astype("float32")
df["MonthSin"] = pd.to_numeric(df["MonthSin"], errors="coerce").fillna(0.0).astype("float32")
df["MonthCos"] = pd.to_numeric(df["MonthCos"], errors="coerce").fillna(0.0).astype("float32")
df["Summer"] = pd.to_numeric(df["Summer"], errors="coerce").fillna(0.0).astype("float32")
df["Fall"] = pd.to_numeric(df["Fall"], errors="coerce").fillna(0.0).astype("float32")
df["Winter"] = pd.to_numeric(df["Winter"], errors="coerce").fillna(0.0).astype("float32")
df["Spring"] = pd.to_numeric(df["Spring"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0).astype("float32")
df["CDD"] = pd.to_numeric(df["CDD"], errors="coerce").fillna(0.0).astype("float32")
df["HDD"] = pd.to_numeric(df["HDD"], errors="coerce").fillna(0.0).astype("float32")
df["wind_speed"] = pd.to_numeric(df["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_2"] = pd.to_numeric(df["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_6"] = pd.to_numeric(df["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_48"] = pd.to_numeric(df["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_144"] = pd.to_numeric(df["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_2"] = pd.to_numeric(df["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_6"] = pd.to_numeric(df["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_48"] = pd.to_numeric(df["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_144"] = pd.to_numeric(df["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")

# Preserve the categorical feature handling used by the trained baselines.
STATIC_CAT_CODE_COLS = []
if "NEAREST_STATION_NO" in df.columns:
    code_col = static_cat_code_col("NEAREST_STATION_NO")
    codes, _ = pd.factorize(df["NEAREST_STATION_NO"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "STATION_NO" in df.columns:
    code_col = static_cat_code_col("STATION_NO")
    codes, _ = pd.factorize(df["STATION_NO"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "station_id" in df.columns:
    code_col = static_cat_code_col("station_id")
    codes, _ = pd.factorize(df["station_id"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "StationID" in df.columns:
    code_col = static_cat_code_col("StationID")
    codes, _ = pd.factorize(df["StationID"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "TrialRegion" in df.columns:
    code_col = static_cat_code_col("TrialRegion")
    codes, _ = pd.factorize(df["TrialRegion"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "TRIAL_REGION" in df.columns:
    code_col = static_cat_code_col("TRIAL_REGION")
    codes, _ = pd.factorize(df["TRIAL_REGION"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "TrialRegionID" in df.columns:
    code_col = static_cat_code_col("TrialRegionID")
    codes, _ = pd.factorize(df["TrialRegionID"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "trial_region_id" in df.columns:
    code_col = static_cat_code_col("trial_region_id")
    codes, _ = pd.factorize(df["trial_region_id"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
STATIC_COLS = STATIC_BASE_COLS + STATIC_CAT_CODE_COLS

# Sum interval energy, average time-varying fields, and preserve household attributes.
aggregation = {
    TARGET_COL: "sum",
    "HourSin": "mean",
    "HourCos": "mean",
    "HalfHourSin": "mean",
    "HalfHourCos": "mean",
    "WeekdaySin": "mean",
    "WeekdayCos": "mean",
    "MonthSin": "mean",
    "MonthCos": "mean",
    "Summer": "mean",
    "Fall": "mean",
    "Winter": "mean",
    "Spring": "mean",
    "Temperature": "mean",
    "CDD": "mean",
    "HDD": "mean",
    "wind_speed": "mean",
    "Temperature_lag_2": "mean",
    "Temperature_lag_6": "mean",
    "Temperature_lag_48": "mean",
    "Temperature_lag_144": "mean",
    "temperature_lag_2": "mean",
    "temperature_lag_6": "mean",
    "temperature_lag_48": "mean",
    "temperature_lag_144": "mean",
    "NUM_OCCUPANTS": "first",
    "NUM_ROOMS_HEATED": "first",
    "NUM_REFRIGERATORS": "first",
    "Unit": "first",
    "SemiDetached": "first",
    "SeparateHouse": "first",
    "HAS_GAS_HEATING": "first",
    "HAS_GAS_HOT_WATER": "first",
    "HAS_GAS_COOKING": "first",
    "HAS_POOLPUMP": "first",
    "Ducted": "first",
    "SplitSystem": "first",
    "NoAirCon": "first",
    "OtherAirCon": "first",
    "CONTROLLED_LOAD_CNT": "first",
}
for code_col in STATIC_CAT_CODE_COLS:
    aggregation[code_col] = "first"

df = (
    df.set_index(TIME_COL)
      .groupby(HOUSEHOLD_ID_COL)
      .resample(pandas_freq(FREQ))
      .agg(aggregation)
      .reset_index()
      .sort_values([HOUSEHOLD_ID_COL, TIME_COL])
      .reset_index(drop=True)
)

# Calendar values are recalculated from the final resampled timestamps.
df = add_calendar_covariates(df, df[TIME_COL])

# Split complete households so no customer's windows appear in both partitions.
rng = np.random.default_rng(SEED)
all_customers = np.array(sorted(df[HOUSEHOLD_ID_COL].dropna().unique()))
n_val = max(1, int(len(all_customers) * VAL_FRAC))
val_customers = set(rng.choice(all_customers, size=n_val, replace=False).tolist())
train_customers = set(all_customers.tolist()) - val_customers

print("Preprocessed dataframe shape:", df.shape)
print(
    f"Customers total={len(all_customers)} | "
    f"train={len(train_customers)} | val={len(val_customers)}"
)
print("Past covariates:", len(PAST_COV_COLS), "| Future covariates:", len(FUTURE_COV_COLS), "| Static features:", len(STATIC_COLS))


In [ ]:
# These seven metrics form the complete evaluation set used by the cleaned notebooks.
METRIC_NAMES = [
    "MAE",
    "RMSE",
    "PeakMAE",
    "QuantileLoss",
    "KL_Divergence",
    "DTW",
    "WinklerScore",
]


def pinball_loss(y_true, prediction, quantile):
    error = np.asarray(y_true, dtype=np.float64) - np.asarray(
        prediction,
        dtype=np.float64,
    )
    return float(
        np.nanmean(
            np.maximum(
                quantile * error,
                (quantile - 1.0) * error,
            )
        )
    )


def finite_flat(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    return values[np.isfinite(values)]


def kl_divergence(real, forecast):
    real_flat = finite_flat(real)
    forecast_flat = finite_flat(forecast)
    lower = float(real_flat.min())
    upper = float(real_flat.max())
    if upper <= lower:
        upper = lower + 1e-6

    edges = np.linspace(lower, upper, BINS + 1)
    real_histogram, _ = np.histogram(real_flat, bins=edges)
    forecast_histogram, _ = np.histogram(forecast_flat, bins=edges)

    epsilon = 1e-8
    p = real_histogram.astype(np.float64) + epsilon
    q = forecast_histogram.astype(np.float64) + epsilon
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))


def dtw_distance(real, forecast, max_points=256):
    real = np.asarray(real, dtype=np.float64).reshape(-1)
    forecast = np.asarray(forecast, dtype=np.float64).reshape(-1)

    if len(real) > max_points:
        indices = np.linspace(0, len(real) - 1, max_points).round().astype(int)
        real = real[indices]
        forecast = forecast[indices]

    previous = np.full(len(forecast) + 1, np.inf)
    current = np.full(len(forecast) + 1, np.inf)
    previous[0] = 0.0

    for row in range(1, len(real) + 1):
        current[0] = np.inf
        for column in range(1, len(forecast) + 1):
            difference = abs(real[row - 1] - forecast[column - 1])
            current[column] = difference + min(
                previous[column],
                current[column - 1],
                previous[column - 1],
            )
        previous, current = current, previous
    return float(previous[-1])


def compute_metrics(y_real, lower, median, upper):
    y_real = np.asarray(y_real, dtype=np.float64)
    lower = np.asarray(lower, dtype=np.float64)
    median = np.asarray(median, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)

    interval_lower = np.minimum(lower, upper)
    interval_upper = np.maximum(lower, upper)

    MAE = float(np.nanmean(np.abs(median - y_real)))
    RMSE = float(np.sqrt(np.nanmean((median - y_real) ** 2)))
    PeakMAE = float(
        np.nanmean(
            np.abs(
                np.nanmax(median[:, :, 0], axis=1)
                - np.nanmax(y_real[:, :, 0], axis=1)
            )
        )
    )

    quantile_losses = [
        pinball_loss(y_real, interval_lower, 0.1),
        pinball_loss(y_real, median, 0.5),
        pinball_loss(y_real, interval_upper, 0.9),
    ]
    QuantileLoss = float(np.mean(quantile_losses))
    KL_Divergence = kl_divergence(y_real, median)

    dtw_count = min(32, y_real.shape[0])
    DTW = float(
        np.nanmean(
            [
                dtw_distance(y_real[index, :, 0], median[index, :, 0])
                for index in range(dtw_count)
            ]
        )
    )

    interval_width = interval_upper - interval_lower
    below = y_real < interval_lower
    above = y_real > interval_upper
    WinklerScore = float(
        np.nanmean(
            interval_width
            + (2.0 / ALPHA_PI) * (interval_lower - y_real) * below
            + (2.0 / ALPHA_PI) * (y_real - interval_upper) * above
        )
    )

    return {
        "MAE": MAE,
        "RMSE": RMSE,
        "PeakMAE": PeakMAE,
        "QuantileLoss": QuantileLoss,
        "KL_Divergence": KL_Divergence,
        "DTW": DTW,
        "WinklerScore": WinklerScore,
    }


def print_metrics_table(rows):
    table = pd.DataFrame(rows)
    table = table[METRIC_NAMES]
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(table.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
    return table


In [ ]:
# N-HiTS baseline implementation from Darts; paper: https://arxiv.org/abs/2201.12886
from darts.models import NHiTSModel

import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.utils.likelihood_models import QuantileRegression


def trainer_kwargs():
    kwargs = {"enable_checkpointing": False, "logger": False, "gradient_clip_val": 1.0}
    if torch.cuda.is_available():
        kwargs.update({"accelerator": "gpu", "devices": 1, "precision": "32-true"})
    else:
        kwargs.update({"accelerator": "cpu", "devices": 1})
    return kwargs


def dataloader_kwargs():
    kwargs = {"num_workers": NUM_LOADER_WORKERS}
    if NUM_LOADER_WORKERS > 0:
        kwargs["persistent_workers"] = True
    if torch.cuda.is_available():
        kwargs["pin_memory"] = True
    return kwargs


def ts_nan_to_zero(series):
    values = series.all_values(copy=True)
    values[~np.isfinite(values)] = 0.0
    return series.with_values(values)


past_cov_cols = list(PAST_COV_COLS)
future_cov_cols = list(FUTURE_COV_COLS)
static_cols = list(STATIC_COLS)

y_train = []
y_val = []
pc_train = []
pc_val = []
fc_train = []
fc_val = []

# Convert one complete customer at a time so sequences cannot cross household boundaries.
for household_id, customer in df.groupby(HOUSEHOLD_ID_COL):
    customer = customer.sort_values(TIME_COL).drop_duplicates(TIME_COL, keep="last")
    if customer.empty:
        continue

    complete_index = pd.date_range(
        customer[TIME_COL].min(),
        customer[TIME_COL].max(),
        freq=pandas_freq(FREQ),
    )
    customer = customer.set_index(TIME_COL).reindex(complete_index)
    customer.index.name = TIME_COL
    customer[HOUSEHOLD_ID_COL] = household_id

    # Static values are propagated over any missing timestamps.
    customer["NUM_OCCUPANTS"] = pd.to_numeric(customer["NUM_OCCUPANTS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["NUM_ROOMS_HEATED"] = pd.to_numeric(customer["NUM_ROOMS_HEATED"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["NUM_REFRIGERATORS"] = pd.to_numeric(customer["NUM_REFRIGERATORS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["Unit"] = pd.to_numeric(customer["Unit"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["SemiDetached"] = pd.to_numeric(customer["SemiDetached"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["SeparateHouse"] = pd.to_numeric(customer["SeparateHouse"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_GAS_HEATING"] = pd.to_numeric(customer["HAS_GAS_HEATING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_GAS_HOT_WATER"] = pd.to_numeric(customer["HAS_GAS_HOT_WATER"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_GAS_COOKING"] = pd.to_numeric(customer["HAS_GAS_COOKING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_POOLPUMP"] = pd.to_numeric(customer["HAS_POOLPUMP"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["Ducted"] = pd.to_numeric(customer["Ducted"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["SplitSystem"] = pd.to_numeric(customer["SplitSystem"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["NoAirCon"] = pd.to_numeric(customer["NoAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["OtherAirCon"] = pd.to_numeric(customer["OtherAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["CONTROLLED_LOAD_CNT"] = pd.to_numeric(customer["CONTROLLED_LOAD_CNT"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    for code_col in STATIC_CAT_CODE_COLS:
        customer[code_col] = pd.to_numeric(customer[code_col], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")

    # Calendar values come from the completed timestamp grid; other temporal gaps become zero.
    customer = add_calendar_covariates(customer, customer.index)
    customer[TARGET_COL] = pd.to_numeric(customer[TARGET_COL], errors="coerce").fillna(0.0).astype("float32")
    customer["HourSin"] = pd.to_numeric(customer["HourSin"], errors="coerce").fillna(0.0).astype("float32")
    customer["HourCos"] = pd.to_numeric(customer["HourCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["HalfHourSin"] = pd.to_numeric(customer["HalfHourSin"], errors="coerce").fillna(0.0).astype("float32")
    customer["HalfHourCos"] = pd.to_numeric(customer["HalfHourCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["WeekdaySin"] = pd.to_numeric(customer["WeekdaySin"], errors="coerce").fillna(0.0).astype("float32")
    customer["WeekdayCos"] = pd.to_numeric(customer["WeekdayCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["MonthSin"] = pd.to_numeric(customer["MonthSin"], errors="coerce").fillna(0.0).astype("float32")
    customer["MonthCos"] = pd.to_numeric(customer["MonthCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["Summer"] = pd.to_numeric(customer["Summer"], errors="coerce").fillna(0.0).astype("float32")
    customer["Fall"] = pd.to_numeric(customer["Fall"], errors="coerce").fillna(0.0).astype("float32")
    customer["Winter"] = pd.to_numeric(customer["Winter"], errors="coerce").fillna(0.0).astype("float32")
    customer["Spring"] = pd.to_numeric(customer["Spring"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature"] = pd.to_numeric(customer["Temperature"], errors="coerce").fillna(0.0).astype("float32")
    customer["CDD"] = pd.to_numeric(customer["CDD"], errors="coerce").fillna(0.0).astype("float32")
    customer["HDD"] = pd.to_numeric(customer["HDD"], errors="coerce").fillna(0.0).astype("float32")
    customer["wind_speed"] = pd.to_numeric(customer["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_2"] = pd.to_numeric(customer["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_6"] = pd.to_numeric(customer["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_48"] = pd.to_numeric(customer["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_144"] = pd.to_numeric(customer["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_2"] = pd.to_numeric(customer["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_6"] = pd.to_numeric(customer["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_48"] = pd.to_numeric(customer["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_144"] = pd.to_numeric(customer["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
    customer = customer.reset_index()

    if len(customer) < INPUT_LEN + SEQ_LEN:
        continue

    target_series = TimeSeries.from_dataframe(
        customer,
        time_col=TIME_COL,
        value_cols=TARGET_COL,
        freq=pandas_freq(FREQ),
    )
    target_series = ts_nan_to_zero(target_series)
    target_series = target_series.with_static_covariates(
        customer[static_cols].iloc[[0]].astype("float32")
    )

    past_series = TimeSeries.from_dataframe(
        customer,
        time_col=TIME_COL,
        value_cols=past_cov_cols,
        freq=pandas_freq(FREQ),
    )
    past_series = ts_nan_to_zero(past_series)

    future_series = TimeSeries.from_dataframe(
        customer,
        time_col=TIME_COL,
        value_cols=future_cov_cols,
        freq=pandas_freq(FREQ),
    )
    future_series = ts_nan_to_zero(future_series)

    if household_id in val_customers:
        y_val.append(target_series)
        pc_val.append(past_series)
        fc_val.append(future_series)
    else:
        y_train.append(target_series)
        pc_train.append(past_series)
        fc_train.append(future_series)

if not y_train or not y_val:
    raise RuntimeError("The customer split did not produce usable train and validation series.")

prepared = {
    "y_train": y_train,
    "y_val": y_val,
    "pc_train": pc_train,
    "pc_val": pc_val,
    "fc_train": fc_train,
    "fc_val": fc_val,
    "past_cov_cols": past_cov_cols,
    "future_cov_cols": future_cov_cols,
    "static_cols": static_cols,
}

print(f"Train series={len(y_train)} | Validation series={len(y_val)}")
print("Past covariates:", past_cov_cols)
print("Future covariates:", future_cov_cols)
print("Static covariates:", static_cols)

# Fit every scaler on training households only, then reuse it for validation.
y_scaler = Scaler(MinMaxScaler(), global_fit=True)
pc_scaler = Scaler(StandardScaler(), global_fit=True)
fc_scaler = Scaler(StandardScaler(), global_fit=True)

y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled = y_scaler.transform(y_val)
pc_train_scaled = pc_scaler.fit_transform(pc_train)
pc_val_scaled = pc_scaler.transform(pc_val)
fc_train_scaled = fc_scaler.fit_transform(fc_train)
fc_val_scaled = fc_scaler.transform(fc_val)


In [ ]:

torch.set_float32_matmul_precision("medium")
# Quantile regression trains direct predictive intervals instead of a single point forecast; source: https://en.wikipedia.org/wiki/Quantile_regression
likelihood = QuantileRegression(quantiles=QUANTILES)

model = NHiTSModel(
    input_chunk_length=INPUT_LEN,
    output_chunk_length=SEQ_LEN,
    num_stacks=3,
    num_blocks=2,
    num_layers=2,
    layer_widths=128,
    dropout=0.1,
    batch_size=BATCH_SIZE,
    n_epochs=EPOCHS,
    likelihood=likelihood,
    random_state=SEED,
    pl_trainer_kwargs=trainer_kwargs(),
)


In [ ]:
# Train a new model. Run this cell before the save cell.
model.fit(
    series=y_train_scaled,
    past_covariates=pc_train_scaled,
    future_covariates=fc_train_scaled if False else None,
    verbose=True,
    max_samples_per_ts=MAX_TRAIN_SAMPLES_PER_TS,
    stride=TRAIN_STRIDE,
    dataloader_kwargs=dataloader_kwargs(),
)



In [ ]:
# Save the model produced by the training cell.
checkpoint_name = (
    f"baseline-{MODEL_TAG}__freq-{FREQ}__hor-{PRIMARY_HORIZON}__seq-{SEQ_LEN}"
    f"__seed-{SEED}__ep-{EPOCHS}__bs-{BATCH_SIZE}__in-{INPUT_LEN}"
    f"__stride-{TRAIN_STRIDE}__maxsamp-{MAX_TRAIN_SAMPLES_PER_TS}__q-10-50-90"
)
checkpoint_dir = MODEL_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pt"
model.save(str(checkpoint_path))
print("Saved:", checkpoint_path)


In [ ]:
# Select the checkpoint to load, then run this cell instead of the training and save cells.
CHECKPOINT_TO_LOAD = (
    Path("..") / "checkpoints" / "N-HiTS" /
    "baseline-nhits__freq-30min__hor-24h__seq-48__seed-0__ep-10__bs-64__in-96__stride-48__maxsamp-64__q-10-50-90.pt"
)

# Loads the Darts model and its adjacent .pt.ckpt weight file.
def load_checkpoint(path):
    selected_path = Path(path)
    companion_path = Path(f"{selected_path}.ckpt")
    if not selected_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {selected_path}")
    if not companion_path.is_file():
        raise FileNotFoundError(f"Darts companion checkpoint not found: {companion_path}")

    loaded_model = NHiTSModel.load(
        str(selected_path),
        pl_trainer_kwargs=trainer_kwargs(),
    )
    dimensions = {
        "INPUT_LEN": (loaded_model.input_chunk_length, INPUT_LEN),
        "SEQ_LEN": (loaded_model.output_chunk_length, SEQ_LEN),
    }
    mismatches = {
        key: values for key, values in dimensions.items() if values[0] != values[1]
    }
    loaded_quantiles = list(getattr(loaded_model.likelihood, "quantiles", []))
    if len(loaded_quantiles) != len(QUANTILES) or not np.allclose(loaded_quantiles, QUANTILES):
        mismatches["QUANTILES"] = (loaded_quantiles, QUANTILES)
    expected_name_parts = (
        f"__freq-{FREQ}__",
        f"__hor-{PRIMARY_HORIZON}__",
        f"__seq-{SEQ_LEN}__",
        f"__in-{INPUT_LEN}__",
    )
    if not all(part in selected_path.name for part in expected_name_parts):
        mismatches["filename"] = (selected_path.name, expected_name_parts)
    if mismatches:
        details = "; ".join(
            f"{key}: saved={saved!r}, current={current!r}"
            for key, (saved, current) in mismatches.items()
        )
        raise ValueError(f"Checkpoint settings do not match this run: {details}")

    print("Loaded:", selected_path)
    return loaded_model

model = load_checkpoint(CHECKPOINT_TO_LOAD)


In [ ]:
USE_FUTURE_COVARIATES = False


def quantile_values(ts, q):
    vals = ts.quantile(q).values(copy=False)
    if vals.ndim == 1:
        vals = vals[:, None]
    return vals.astype("float32")


# Chooses validation start positions without evaluating every possible overlapping window.
def eval_start_positions(series_length):
    max_start = series_length - SEQ_LEN
    if max_start < INPUT_LEN:
        return []
    return list(range(INPUT_LEN, max_start + 1, SEQ_LEN))


# Runs rolling validation forecasts for Darts baselines and returns arrays used by the metric table.
def evaluate_darts_model(eval_pool):
    if eval_pool == "train":
        y_pool = y_train_scaled
        pc_pool = pc_train_scaled
        fc_pool = fc_train_scaled
        seed_offset = 1000
    elif eval_pool == "validation":
        y_pool = y_val_scaled
        pc_pool = pc_val_scaled
        fc_pool = fc_val_scaled
        seed_offset = 2000
    else:
        raise ValueError("eval_pool must be 'train' or 'validation'")

    y_real, q10s, q50s, q90s = [], [], [], []
    rng_eval = np.random.default_rng(SEED + seed_offset)
    series_order = np.arange(len(y_pool))
    rng_eval.shuffle(series_order)
    for idx in series_order:
        if len(y_real) >= EVAL_WINDOWS:
            break
        ys = y_pool[idx]
        starts = eval_start_positions(len(ys))
        if not starts:
            continue
        rng_eval.shuffle(starts)
        for forecast_start in starts:
            if len(y_real) >= EVAL_WINDOWS:
                break
            history_end = ys.time_index[forecast_start - 1]
            truth_start = ys.time_index[forecast_start]
            truth_end = ys.time_index[forecast_start + SEQ_LEN - 1]
            history = ys.slice(ys.start_time(), history_end)
            truth_scaled = ys.slice(truth_start, truth_end)
            pred_kwargs = {
                "n": SEQ_LEN,
                "series": history,
                "num_samples": NUM_SAMPLES,
                "batch_size": BATCH_SIZE,
                "verbose": False,
                "show_warnings": False,
                "dataloader_kwargs": dataloader_kwargs(),
                "random_state": SEED + seed_offset + len(y_real),
            }
            if pc_pool is not None:
                pred_kwargs["past_covariates"] = pc_pool[idx]
            if USE_FUTURE_COVARIATES and fc_pool is not None:
                pred_kwargs["future_covariates"] = fc_pool[idx]
            pred_scaled = model.predict(**pred_kwargs)
            if len(truth_scaled) != SEQ_LEN or len(pred_scaled) != SEQ_LEN:
                continue
            truth = y_scaler.inverse_transform(truth_scaled)
            pred = y_scaler.inverse_transform(pred_scaled)
            real = np.maximum(truth.values(copy=False).astype("float32"), 0.0)
            q10 = np.maximum(quantile_values(pred, 0.1), 0.0)
            q50 = np.maximum(quantile_values(pred, 0.5), 0.0)
            q90 = np.maximum(quantile_values(pred, 0.9), 0.0)
            y_real.append(real)
            q10s.append(np.minimum(q10, q90))
            q50s.append(q50)
            q90s.append(np.maximum(q10, q90))
    if not y_real:
        raise RuntimeError(f"No {eval_pool} forecasts were produced. Try reducing INPUT_LEN or checking series length.")
    return np.stack(y_real, axis=0), np.stack(q10s, axis=0), np.stack(q50s, axis=0), np.stack(q90s, axis=0)

metric_rows = []
for eval_pool in ("validation",):
    y_real_kwh, y_q10_kwh, y_q50_kwh, y_q90_kwh = evaluate_darts_model(eval_pool)
    metrics = compute_metrics(y_real_kwh, y_q10_kwh, y_q50_kwh, y_q90_kwh)
    metric_rows.append(metrics)

metrics_table = print_metrics_table(metric_rows)
